In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:
import dask.dataframe as dd

df = dd.read_parquet("transactions_additional/*/*.parquet")
df.head()

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0000000000,END,NaN,NaN,122.175.102.100,53657.59,CI,<NA>,normal
1,TXN_0000000001,UPD,15.786843,78.077894,103.92.190.204,53479.10,CI,<NA>,normal
2,TXN_0000000002,CCL,NaN,NaN,<NA>,52479.10,CI,<NA>,normal
3,TXN_0000000003,UPC,15.781886,78.053709,122.124.163.67,59479.10,CI,<NA>,normal
4,TXN_0000000004,FTD,NaN,NaN,103.241.164.38,64179.10,CI,<NA>,normal


In [ ]:
import duckdb

con = duckdb.connect()

# Performance tuning
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='10GB'")

# -----------------------------
# Load transaction dataset
# -----------------------------
con.execute("""
CREATE OR REPLACE VIEW txn AS
SELECT
    account_id,
    CAST(transaction_timestamp AS TIMESTAMP) AS ts,
    amount,
    txn_type
FROM read_parquet('transactions/**/*.parquet')
""")

# ---------------------------------------------------
# Base statistics per account
# ---------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE base_stats AS
SELECT
    account_id,

    COUNT(*) AS txn_count,
    SUM(amount) AS total_transaction_amount,
    AVG(amount) AS avg_transaction_amount,
    MAX(amount) AS max_transaction_amount,
    MIN(amount) AS min_transaction_amount,
    STDDEV_POP(amount) AS std_transaction_amount,
    MEDIAN(amount) AS median_transaction_amount,

    SUM(CASE WHEN txn_type='C' THEN amount ELSE 0 END) AS total_credit_amount,
    SUM(CASE WHEN txn_type='D' THEN amount ELSE 0 END) AS total_debit_amount,

    MAX(CASE WHEN txn_type='C' THEN amount ELSE NULL END) AS max_credit_amount,
    MAX(CASE WHEN txn_type='D' THEN amount ELSE NULL END) AS max_debit_amount,

    SUM(CASE WHEN txn_type='C' THEN 1 ELSE 0 END) AS credit_count,
    SUM(CASE WHEN txn_type='D' THEN 1 ELSE 0 END) AS debit_count,

    MIN(ts) AS first_txn,
    MAX(ts) AS last_txn

FROM txn
GROUP BY account_id

""")

# ---------------------------------------------------
# Inter-arrival time features
# ---------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE txn_gaps AS
SELECT
    account_id,
    ts,
    EXTRACT(EPOCH FROM (ts - LAG(ts) OVER
        (PARTITION BY account_id ORDER BY ts))) AS gap_seconds
FROM txn

""")

con.execute("""

CREATE OR REPLACE TABLE gap_stats AS
SELECT
    account_id,
    AVG(gap_seconds) AS avg_time_between_txn_sec,
    MEDIAN(gap_seconds) AS median_time_between_txn_sec
FROM txn_gaps
WHERE gap_seconds IS NOT NULL
GROUP BY account_id

""")

# ---------------------------------------------------
# Burst features
# ---------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE hourly_txn AS
SELECT
    account_id,
    DATE_TRUNC('hour', ts) AS hr,
    COUNT(*) AS txn_count
FROM txn
GROUP BY account_id, hr

""")

con.execute("""

CREATE OR REPLACE TABLE daily_txn AS
SELECT
    account_id,
    DATE_TRUNC('day', ts) AS dy,
    COUNT(*) AS txn_count
FROM txn
GROUP BY account_id, dy

""")

con.execute("""

CREATE OR REPLACE TABLE weekly_txn AS
SELECT
    account_id,
    DATE_TRUNC('week', ts) AS wk,
    COUNT(*) AS txn_count
FROM txn
GROUP BY account_id, wk

""")

con.execute("""

CREATE OR REPLACE TABLE burst_stats AS
SELECT
    account_id,
    MAX(txn_count) AS max_txn_1hr
FROM hourly_txn
GROUP BY account_id

""")

# 10 min burst
con.execute("""

CREATE OR REPLACE TABLE tenmin_txn AS
SELECT
    account_id,
    DATE_TRUNC('minute', ts) -
        INTERVAL (EXTRACT(MINUTE FROM ts) % 10) MINUTE AS ten_min_bucket,
    COUNT(*) AS txn_count
FROM txn
GROUP BY account_id, ten_min_bucket

""")

con.execute("""

CREATE OR REPLACE TABLE tenmin_stats AS
SELECT
    account_id,
    MAX(txn_count) AS max_txn_10min
FROM tenmin_txn
GROUP BY account_id

""")

# Daily burst
con.execute("""

CREATE OR REPLACE TABLE daily_stats AS
SELECT
    account_id,
    MAX(txn_count) AS max_txn_1day
FROM daily_txn
GROUP BY account_id

""")

# Weekly burst
con.execute("""

CREATE OR REPLACE TABLE weekly_stats AS
SELECT
    account_id,
    MAX(txn_count) AS max_txn_1week
FROM weekly_txn
GROUP BY account_id

""")

# ---------------------------------------------------
# Night vs Day transaction rate
# ---------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE day_night AS
SELECT
    account_id,

    SUM(CASE WHEN EXTRACT(HOUR FROM ts) BETWEEN 0 AND 6 THEN 1 ELSE 0 END) AS night_txn,
    SUM(CASE WHEN EXTRACT(HOUR FROM ts) BETWEEN 7 AND 22 THEN 1 ELSE 0 END) AS day_txn,

    COUNT(*) AS total_txn

FROM txn
GROUP BY account_id

""")

# ---------------------------------------------------
# Rapid pass-through detection
# credit and debit same amount within a day
# ---------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE rapid_passthrough AS
SELECT
    account_id,
    COUNT(*) AS rapid_passthrough_count
FROM (
    SELECT
        account_id,
        DATE_TRUNC('day', ts) AS day,
        amount,
        SUM(CASE WHEN txn_type='C' THEN 1 ELSE 0 END) AS c,
        SUM(CASE WHEN txn_type='D' THEN 1 ELSE 0 END) AS d
    FROM txn
    GROUP BY account_id, day, amount
)
WHERE c > 0 AND d > 0
GROUP BY account_id

""")

# ---------------------------------------------------
# Final feature table
# ---------------------------------------------------
transaction_features_2 = con.execute("""

SELECT
    b.account_id,

    b.total_transaction_amount,
    b.avg_transaction_amount,
    b.max_transaction_amount,
    b.min_transaction_amount,
    b.std_transaction_amount,
    b.median_transaction_amount,

    b.total_credit_amount,
    b.total_debit_amount,
    b.max_credit_amount,
    b.max_debit_amount,

    CASE
        WHEN b.total_debit_amount = 0 THEN NULL
        ELSE b.total_credit_amount / b.total_debit_amount
    END AS credit_debit_ratio,

    b.credit_count * 1.0 / b.txn_count AS credit_ratio,
    b.debit_count * 1.0 / b.txn_count AS debit_ratio,

    g.avg_time_between_txn_sec,
    g.median_time_between_txn_sec,

    h.max_txn_1hr,
    t.max_txn_10min,
    d.max_txn_1day,
    w.max_txn_1week,

    b.txn_count / NULLIF(DATE_DIFF('day', b.first_txn, b.last_txn)+1,0) AS avg_txn_per_day,
    b.txn_count / NULLIF(DATE_DIFF('hour', b.first_txn, b.last_txn)+1,0) AS avg_txn_per_hr,
    b.txn_count / NULLIF(DATE_DIFF('week', b.first_txn, b.last_txn)+1,0) AS avg_txn_per_week,
    b.txn_count / NULLIF(DATE_DIFF('month', b.first_txn, b.last_txn)+1,0) AS avg_txn_per_month,

    dn.night_txn * 1.0 / dn.total_txn AS night_txn_rate,
    dn.day_txn * 1.0 / dn.total_txn AS day_txn_rate,

    r.rapid_passthrough_count

FROM base_stats b
LEFT JOIN gap_stats g USING(account_id)
LEFT JOIN burst_stats h USING(account_id)
LEFT JOIN tenmin_stats t USING(account_id)
LEFT JOIN daily_stats d USING(account_id)
LEFT JOIN weekly_stats w USING(account_id)
LEFT JOIN day_night dn USING(account_id)
LEFT JOIN rapid_passthrough r USING(account_id)

""").df()

In [4]:
con.execute("""
COPY (
    SELECT * FROM transaction_features_2
)
TO 'transaction_features_2.parquet'
(FORMAT PARQUET);
""")

In [7]:
import dask.dataframe as dd
dff = dd.read_parquet("transaction_features_2.parquet")    
dff.head()

,account_id,total_transaction_amount,avg_transaction_amount,max_transaction_amount,min_transaction_amount,std_transaction_amount,median_transaction_amount,total_credit_amount,total_debit_amount,max_credit_amount,max_debit_amount,credit_debit_ratio,credit_ratio,debit_ratio,avg_time_between_txn_sec,median_time_between_txn_sec,max_txn_1hr,max_txn_10min,max_txn_1day,max_txn_1week,avg_txn_per_day,avg_txn_per_hr,avg_txn_per_week,avg_txn_per_month,night_txn_rate,day_txn_rate,rapid_passthrough_count
0,ACCT_002631,2.104429e+08,25579.548961,35214000.00,-2270044.26,407942.578626,1000.0,81534279.29,1.289087e+08,4418871.15,35214000.00,0.632496,0.447308,0.552692,981.771699,459.0,17,6,138,673,87.521277,3.664588,587.642857,2056.750000,0.056521,0.926462,323.0
1,ACCT_002643,5.808451e+07,29665.226404,14496908.97,-127127.28,365480.588591,900.0,18665007.33,3.941951e+07,4716000.00,14496908.97,0.473497,0.471399,0.528601,32101.411344,16870.0,3,3,10,30,2.689560,0.112193,18.826923,81.583333,0.050562,0.929520,10.0
2,ACCT_002653,2.815076e+07,21839.222327,3312531.54,-2500.00,142942.038372,800.0,14683386.15,1.346737e+07,3312531.54,2421381.76,1.090293,0.494957,0.505043,122284.975932,82788.0,3,2,5,13,0.706689,0.029462,4.938697,21.483333,0.055081,0.933282,1.0
3,ACCT_002660,4.405927e+07,23880.365599,9148897.93,-52000.00,230541.369695,1000.0,16494792.15,2.756448e+07,1022119.47,9148897.93,0.598407,0.462331,0.537669,29905.380694,16905.0,4,2,9,31,2.887324,0.120431,20.054348,83.863636,0.061247,0.918699,10.0
4,ACCT_002667,8.148030e+07,29341.123554,12829813.10,-36461.27,327845.023800,1000.0,36753995.17,4.472630e+07,9862500.00,12829813.10,0.821753,0.473533,0.526467,26114.803314,13702.5,4,2,11,39,3.305952,0.137899,23.141667,99.178571,0.055456,0.922578,15.0


### **Counterparty id : -**

In [8]:
import duckdb

# connect duckdb
con = duckdb.connect("aml_features.duckdb")

# performance settings
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='10GB'")

# --------------------------------------------------
# Load transaction dataset
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE VIEW txn AS
SELECT
    account_id,
    counterparty_id,
    txn_type,
    amount,
    CAST(transaction_timestamp AS TIMESTAMP) AS ts
FROM read_parquet('transactions/**/*.parquet')

""")

# --------------------------------------------------
# Base counterparty statistics
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE counterparty_base AS
SELECT
    account_id,

    COUNT(*) AS total_txn,

    COUNT(DISTINCT counterparty_id) 
        AS unique_counterparty_count,

    COUNT(*) - COUNT(DISTINCT counterparty_id) 
        AS repeated_counterparty_count,

    (COUNT(*) - COUNT(DISTINCT counterparty_id))*1.0 / COUNT(*) 
        AS repeated_counterparty_frequency,

    COUNT(DISTINCT CASE 
        WHEN txn_type='C' THEN counterparty_id END)
        AS unique_credit_counterparty_count,

    COUNT(DISTINCT CASE 
        WHEN txn_type='D' THEN counterparty_id END)
        AS unique_debit_counterparty_count

FROM txn
GROUP BY account_id

""")

# --------------------------------------------------
# Hourly counterparty activity
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE hourly_counterparty AS
SELECT
    account_id,
    DATE_TRUNC('hour', ts) AS hr,

    COUNT(*) AS txn_count,

    COUNT(DISTINCT counterparty_id) 
        AS unique_counterparty_hour

FROM txn
GROUP BY account_id, hr

""")

# --------------------------------------------------
# Daily counterparty activity
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE daily_counterparty AS
SELECT
    account_id,
    DATE_TRUNC('day', ts) AS dy,

    COUNT(*) AS txn_count,

    COUNT(DISTINCT counterparty_id) 
        AS unique_counterparty_day

FROM txn
GROUP BY account_id, dy

""")

# --------------------------------------------------
# Hourly statistics
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE hourly_counterparty_stats AS
SELECT
    account_id,

    AVG(unique_counterparty_hour)
        AS avg_unique_counterparty_1hr,

    MAX(unique_counterparty_hour)
        AS max_unique_counterparty_1hr,

    AVG((txn_count - unique_counterparty_hour)*1.0 / txn_count)
        AS avg_repeated_counterparty_freq_1hr

FROM hourly_counterparty
GROUP BY account_id

""")

# --------------------------------------------------
# Daily statistics
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE daily_counterparty_stats AS
SELECT
    account_id,

    AVG(unique_counterparty_day)
        AS avg_unique_counterparty_1day,

    MAX(unique_counterparty_day)
        AS max_unique_counterparty_1day,

    AVG((txn_count - unique_counterparty_day)*1.0 / txn_count)
        AS avg_repeated_counterparty_freq_1day

FROM daily_counterparty
GROUP BY account_id

""")

# --------------------------------------------------
# Final AML feature table
# --------------------------------------------------
con.execute("""

CREATE OR REPLACE TABLE transaction_features_3 AS
SELECT
    b.account_id,

    b.unique_counterparty_count,
    b.repeated_counterparty_frequency,

    b.unique_credit_counterparty_count,
    b.unique_debit_counterparty_count,

    h.avg_unique_counterparty_1hr,
    h.max_unique_counterparty_1hr,
    h.avg_repeated_counterparty_freq_1hr,

    d.avg_unique_counterparty_1day,
    d.max_unique_counterparty_1day,
    d.avg_repeated_counterparty_freq_1day

FROM counterparty_base b
LEFT JOIN hourly_counterparty_stats h USING(account_id)
LEFT JOIN daily_counterparty_stats d USING(account_id)

""")

# --------------------------------------------------
# Export features to parquet
# --------------------------------------------------
con.execute("""

COPY transaction_features_3
TO 'transaction_features_3.parquet'
(FORMAT PARQUET)

""")

print("AML counterparty features created successfully")

AML counterparty features created successfully


In [3]:
import dask.dataframe as dd
dff = dd.read_parquet("transaction_features_3.parquet")    
dff.head()

,account_id,unique_counterparty_count,repeated_counterparty_frequency,unique_credit_counterparty_count,unique_debit_counterparty_count,avg_unique_counterparty_1hr,max_unique_counterparty_1hr,avg_repeated_counterparty_freq_1hr,avg_unique_counterparty_1day,max_unique_counterparty_1day,avg_repeated_counterparty_freq_1day
0,ACCT_058176,28,0.928021,28,27,1.007772,2,0.000000,1.257329,4,0.003800
1,ACCT_058195,20,0.985785,20,19,1.054929,3,0.001630,1.998519,8,0.025474
2,ACCT_058217,17,0.991935,16,17,1.077160,3,0.003601,2.544503,7,0.051629
3,ACCT_061728,7,0.997389,7,6,1.098292,4,0.008781,2.823188,7,0.213988
4,ACCT_061733,11,0.997950,10,11,1.145214,5,0.008925,3.897170,9,0.184259


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [4]:
import dask.dataframe as dd
dff = dd.read_parquet("transaction_features_2.parquet")    
dff.head()

,account_id,total_transaction_amount,avg_transaction_amount,max_transaction_amount,min_transaction_amount,std_transaction_amount,median_transaction_amount,total_credit_amount,total_debit_amount,max_credit_amount,...,max_txn_10min,max_txn_1day,max_txn_1week,avg_txn_per_day,avg_txn_per_hr,avg_txn_per_week,avg_txn_per_month,night_txn_rate,day_txn_rate,rapid_passthrough_count
0,ACCT_002631,2.104429e+08,25579.548961,35214000.00,-2270044.26,407942.578626,1000.0,81534279.29,1.289087e+08,4418871.15,...,6,138,673,87.521277,3.664588,587.642857,2056.750000,0.056521,0.926462,323.0
1,ACCT_002643,5.808451e+07,29665.226404,14496908.97,-127127.28,365480.588591,900.0,18665007.33,3.941951e+07,4716000.00,...,3,10,30,2.689560,0.112193,18.826923,81.583333,0.050562,0.929520,10.0
2,ACCT_002653,2.815076e+07,21839.222327,3312531.54,-2500.00,142942.038372,800.0,14683386.15,1.346737e+07,3312531.54,...,2,5,13,0.706689,0.029462,4.938697,21.483333,0.055081,0.933282,1.0
3,ACCT_002660,4.405927e+07,23880.365599,9148897.93,-52000.00,230541.369695,1000.0,16494792.15,2.756448e+07,1022119.47,...,2,9,31,2.887324,0.120431,20.054348,83.863636,0.061247,0.918699,10.0
4,ACCT_002667,8.148030e+07,29341.123554,12829813.10,-36461.27,327845.023800,1000.0,36753995.17,4.472630e+07,9862500.00,...,2,11,39,3.305952,0.137899,23.141667,99.178571,0.055456,0.922578,15.0


In [5]:
dff.shape[0].compute()

159776

In [7]:
import dask.dataframe as dd

df = dd.read_parquet("transactions_additional/*/*.parquet")
df.head()

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0000000000,END,NaN,NaN,122.175.102.100,53657.59,CI,<NA>,normal
1,TXN_0000000001,UPD,15.786843,78.077894,103.92.190.204,53479.10,CI,<NA>,normal
2,TXN_0000000002,CCL,NaN,NaN,<NA>,52479.10,CI,<NA>,normal
3,TXN_0000000003,UPC,15.781886,78.053709,122.124.163.67,59479.10,CI,<NA>,normal
4,TXN_0000000004,FTD,NaN,NaN,103.241.164.38,64179.10,CI,<NA>,normal
